In [ ]:
import sys
sys.path.insert(0, "..")

from econometrics.data import RAW_POSITIONS, DAILY_AT_MAP, DAILY_AT_NULL_MAP, IL_MAP
from backtest.oracle_comparison import positions_from_raw_data, build_dual_series
from backtest.mechanism_sweep import run_mechanism_sweep, run_payoff_comparison, build_mechanism_series
from backtest.daily import build_daily_states
from backtest.payoff import run_exit_payoff_backtest
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.figsize"] = (14, 6)

exits = positions_from_raw_data(RAW_POSITIONS)
dual = build_dual_series(exits)

raw_dicts = [{"burn_date": bd, "blocklife": bl} for bd, bl, _ in RAW_POSITIONS]
daily_states = build_daily_states(
    DAILY_AT_MAP, DAILY_AT_NULL_MAP, IL_MAP, raw_dicts, pool_daily_fee=50_000.0
)

print(f"Total exits: {len(exits)}")
print(f"Days with exits: {len(dual.days)}")
print(f"Date range: {dual.days[0]} to {dual.days[-1]}")

## Phase 1: Cumulative vs Daily-Snapshot delta_plus

In [ ]:
fig, ax = plt.subplots()
ax.plot(range(len(dual.days)), dual.cumulative_delta_plus, "r-o", label="Cumulative (Solidity)", markersize=4)
ax.plot(range(len(dual.days)), dual.daily_snapshot_delta_plus, "b-s", label="Daily Snapshot (Backtest)", markersize=4)
ax.axvline(x=dual.days.index("2025-12-23"), color="gray", linestyle="--", alpha=0.5, label="Dec 23 spike")
ax.set_xticks(range(len(dual.days)))
ax.set_xticklabels(dual.days, rotation=45, ha="right", fontsize=7)
ax.set_ylabel("delta_plus")
ax.set_title("FCI Oracle: Cumulative vs Daily-Snapshot delta_plus")
ax.legend()
plt.tight_layout()
plt.show()

dec23_idx = dual.days.index("2025-12-23")
cum_23 = dual.cumulative_delta_plus[dec23_idx]
snap_23 = dual.daily_snapshot_delta_plus[dec23_idx]
print(f"Dec 23 -- Cumulative: {cum_23:.6f}, Daily: {snap_23:.6f}")
if dec23_idx + 1 < len(dual.days):
    cum_24 = dual.cumulative_delta_plus[dec23_idx + 1]
    snap_24 = dual.daily_snapshot_delta_plus[dec23_idx + 1]
    print(f"Dec 24 -- Cumulative: {cum_24:.6f}, Daily: {snap_24:.6f}")
    ratio = cum_24 / max(snap_24, 1e-10)
    print(f"Ratio (cum/daily): {ratio:.2f}x")

## Phase 1: Payoff with Daily-Snapshot Baseline

In [ ]:
baseline_payoff = run_exit_payoff_backtest(daily_states, raw_dicts, gamma=0.10, alpha=2.0, delta_star=0.09)

print("=== Payoff with Daily-Snapshot (Backtest Baseline) ===")
print(f"  % better off:     {baseline_payoff.pct_better_off:.1%}")
print(f"  Mean hedge value:  ${baseline_payoff.mean_hedge_value:.2f}")
print(f"  Total premiums:    ${baseline_payoff.total_premiums:.2f}")
print(f"  Total payouts:     ${baseline_payoff.total_payouts:.2f}")

## Phase 2: Mechanism Sweep Results

In [ ]:
comparisons = run_payoff_comparison(
    exits=exits,
    daily_snapshot_baseline=list(dual.daily_snapshot_delta_plus),
    baseline_days=list(dual.days),
    daily_states=daily_states,
    raw_positions=raw_dicts,
    gamma=0.10,
    alpha=2.0,
    delta_star=0.09,
    epoch_lengths=[1, 3, 7, 14],
    half_lives=[1.0, 3.0, 7.0, 14.0],
    window_sizes=[10, 25, 50],
)

print("=== Mechanism Sweep Results (sorted by correlation) ===")
header = f'{"Mechanism":<10} {"Params":<30} {"Corr":>6} {"MaxDiv":>8} {"%Better":>8} {"MeanHV":>10}'
print(header)
print("-" * 80)
for c in comparisons:
    params_str = ", ".join(f"{k}={v}" for k, v in c.params.items())
    print(f"{c.mechanism_name:<10} {params_str:<30} {c.correlation:>6.3f} {c.max_divergence:>8.4f} {c.pct_better_off:>7.1%} {c.mean_hedge_value:>10.2f}")

viable = [c for c in comparisons if c.correlation > 0.8 and c.mean_hedge_value >= 0]
print(f"\nViable candidates (corr > 0.8 AND mean HV >= 0): {len(viable)}")
for c in viable:
    params_str = ", ".join(f"{k}={v}" for k, v in c.params.items())
    print(f"  {c.mechanism_name} ({params_str}): corr={c.correlation:.3f}, HV=${c.mean_hedge_value:.2f}")

## Phase 2: Top Candidates vs Baseline

In [ ]:
top_n = min(3, len(comparisons))
fig, axes = plt.subplots(top_n, 1, figsize=(14, 4 * top_n), sharex=True)
if top_n == 1:
    axes = [axes]

for i, c in enumerate(comparisons[:top_n]):
    ax = axes[i]
    series = build_mechanism_series(exits, c.mechanism_name, **c.params)
    ax.plot(range(len(dual.days)), dual.daily_snapshot_delta_plus, "b-s",
            label="Daily Snapshot (target)", markersize=3, alpha=0.7)
    mech_dict = dict(zip(series.days, series.delta_plus_values))
    aligned = [mech_dict.get(d, 0) for d in dual.days]
    ax.plot(range(len(dual.days)), aligned, "g-^",
            label=f"{c.mechanism_name} ({c.params})", markersize=3)
    ax.set_ylabel("delta_plus")
    title = f"#{i+1}: {c.mechanism_name} -- corr={c.correlation:.3f}, HV=${c.mean_hedge_value:.2f}"
    ax.set_title(title)
    ax.legend(fontsize=8)

axes[-1].set_xticks(range(len(dual.days)))
axes[-1].set_xticklabels(dual.days, rotation=45, ha="right", fontsize=7)
plt.tight_layout()
plt.show()

## Summary

In [ ]:
print("=" * 60)
print("PHASE 2 SUMMARY")
print("=" * 60)
print(f"Baseline (daily-snapshot): % better off = {baseline_payoff.pct_better_off:.1%}, mean HV = ${baseline_payoff.mean_hedge_value:.2f}")
print()

if viable:
    best = viable[0]
    print(f"BEST CANDIDATE: {best.mechanism_name}")
    params_str = ", ".join(f"{k}={v}" for k, v in best.params.items())
    print(f"  Parameters: {params_str}")
    print(f"  Correlation to baseline: {best.correlation:.4f}")
    print(f"  Max divergence: {best.max_divergence:.6f}")
    print(f"  % better off: {best.pct_better_off:.1%}")
    print(f"  Mean hedge value: ${best.mean_hedge_value:.2f}")
    print()
    if len(viable) > 1:
        print("AMBIGUOUS -- multiple viable candidates. All will ship as parallel oracle APIs.")
        for v in viable:
            vp = ", ".join(f"{k}={vv}" for k, vv in v.params.items())
            print(f"  {v.mechanism_name}({vp}): corr={v.correlation:.3f}")
    else:
        print("CLEAR WINNER -- vault defaults to this mechanism.")
else:
    print("NO VIABLE CANDIDATES -- all fail viability threshold.")
    print("Action: revisit oracle architecture (event-level redesign needed).")